In [1]:
import numpy as np
from scipy.io import loadmat
import sys

np.set_printoptions(
    precision=4,        # 4 Nachkommastellen
    suppress=True,      # Keine wissenschaftliche Notation (1e-04) wenn möglich
    linewidth=150,      # Breitere Zeilen, damit Matrizen nicht so früh umbrechen
    formatter={'float_kind': lambda x: "{: .4f}".format(x)} 
)

In [2]:
# %% Matrizen initialisieren
data = loadmat('data.mat')
A = data['A']
b = data['b']

print(A, A.shape)
print(b, b.shape)

[[ 4 -1 -1  1]
 [-2  6  1  2]
 [-1  1  7 -1]
 [-2  1 -1  5]] (4, 4)
[[-2.0675]
 [ 16.1517]
 [ 17.0363]
 [-6.0032]] (4, 1)


In [3]:
# Sicherstellen, dass b ein Spaltenvektor ist (N, 1)
if b.ndim == 1:
    b = b.reshape(-1, 1)

rows = A.shape[0]

D = np.diag(np.diag(A))

# Oberes/Unteres Dreieck (Upper/Lower triangle)
# Hinweis: In NumPy beinhalten triu/tril standardmäßig die Diagonale (k=0)
# Alternativ kann man daher auch mit D subtrahieren, um strikte Dreiecksmatrizen zu erhalten.
U = np.triu(A, k=1)  # np.triu(A) - D
L = np.tril(A, k=-1) # np.tril(A) - D

In [4]:
# %% Konvergenz prüfen

# Eigenwerte der Iterationsmatrix berechnen: M = -(D+L)^-1 * U
# Hinweis: Der @-Operator steht in Python für Matrixmultiplikation
M = -np.linalg.inv(D + L) @ U
EV = np.linalg.eigvals(M)
absEV = np.abs(EV)

print("Betrag der Eigenwerte:")
print(absEV)
print("-" * 30)

if np.any(absEV > 1):
    raise RuntimeError("Verfahren konvergiert nicht")
else:
    print("Verfahren konvergiert")

Betrag der Eigenwerte:
[ 0.0000  0.1931  0.1931  0.0638]
------------------------------
Verfahren konvergiert


In [5]:
# %% Gauß-Seidel-Algorithmus

x_old = np.zeros((rows, 1))
x     = np.ones((rows, 1)) * -1

fThreshold = 1e-2
nMaxIter   = 5000

# Hilfsfunktion, um die Formatierung einheitlich zu halten
def print_status(iter_curr, iter_prev, diff_norm, x_vec):
    # Vektor als String formatieren
    vec_str = " ".join([f"{val: .8f}" for val in x_vec.flatten()])
    print(f"||x_{iter_curr} - x_{iter_prev}|| = {diff_norm:.4e} : x_{iter_curr}^T = [ {vec_str} ]")

# Initialzustand ausgeben (Iteration 0)
print_status(0, 0, np.linalg.norm(x_old - x), x)

converged = False

for nIterCnt in range(1, nMaxIter + 1):
    # WICHTIG: In Python sind Arrays Referenzen. .copy() erstellt eine echte Kopie.
    x_old = x.copy() 
    
    # Gauß-Seidel Schritt: x = -(D+L)^-1 * U * x_old + (D+L)\b
    # np.linalg.solve(A, b) entspricht dem Matlab Backslash-Operator A\b
    term1 = -np.linalg.inv(D + L) @ U @ x_old
    term2 = np.linalg.solve(D + L, b)
    
    x = term1 + term2
    
    # Norm der Differenz berechnen
    diff_norm = np.linalg.norm(x_old - x)
    
    # Ausgabe des aktuellen Schritts
    print_status(nIterCnt, nIterCnt - 1, diff_norm, x)
    
    # Abbruchkriterium prüfen
    if diff_norm <= fThreshold:
        converged = True
        break

# %% Ergebnisse überprüfen
print('\n***Results***')
print('A*x\t\tb')
# Spalten horizontal stapeln (hstack) für die Ausgabe
Ax = A @ x
print(np.hstack((Ax, b)))

print('\n***Results***')
print('x\t\tinv(A)*b')
# Exakte Lösung mit Solver berechnen (entspricht A\b)
print(np.hstack(
    (
        x, 
        np.linalg.solve(A, b),
    )
))

||x_0 - x_0|| = 2.0000e+00 : x_0^T = [ -1.00000000 -1.00000000 -1.00000000 -1.00000000 ]
||x_1 - x_0|| = 4.8711e+00 : x_1^T = [ -0.76687359  2.93633317  1.76186621 -1.74227338 ]
||x_2 - x_1|| = 2.0255e+00 : x_2^T = [  1.09324460  3.34348599  1.86339364 -1.05935118 ]
||x_3 - x_2|| = 2.9850e-01 : x_3^T = [  1.04968411  3.08440385  1.99174276 -0.99928912 ]
||x_4 - x_3|| = 7.5436e-02 : x_4^T = [  1.00198535  3.02709206  2.00169634 -1.00491555 ]
||x_5 - x_4|| = 1.1744e-02 : x_5^T = [  0.99155240  3.02383096  1.99986801 -1.00880218 ]
||x_6 - x_5|| = 1.8276e-03 : x_6^T = [  0.99125170  3.02533099  1.99905554 -1.00938496 ]

***Results***
A*x		b
[[-2.0688 -2.0675]
 [ 16.1498  16.1517]
 [ 17.0369  17.0363]
 [-6.0032 -6.0032]]

***Results***
x		inv(A)*b
[[ 0.9913  0.9917]
 [ 3.0253  3.0258]
 [ 1.9991  1.9990]
 [-1.0094 -1.0093]]
